# 02 - Silver to Gold
Agregations metier (Batch Processing)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("02_Silver_to_Gold") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

print("Spark connecte")

In [ ]:
# Lire les donnees Silver
trips = spark.read.parquet("hdfs://namenode:9000/projet/silver/trips_clean/")
dates = spark.read.parquet("hdfs://namenode:9000/projet/silver/dates_clean/")
locations = spark.read.parquet("hdfs://namenode:9000/projet/silver/locations_clean/")
weather = spark.read.parquet("hdfs://namenode:9000/projet/silver/weather_clean/")

print(f"Trips Silver: {trips.count()} lignes")

In [ ]:
# Agregation 1 : Stats par borough
stats_borough = (trips
    .join(broadcast(locations),
          trips.pickup_location_id == locations.location_id)
    .groupBy("borough")
    .agg(
        count("*").alias("nb_trajets"),
        round(avg("trip_distance"), 2).alias("distance_moyenne"),
        round(avg("total_amount"), 2).alias("tarif_moyen"),
        round(sum("total_amount"), 2).alias("revenu_total")
    )
    .orderBy(desc("nb_trajets"))
)
stats_borough.show()
stats_borough.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_par_borough/")

In [ ]:
# Agregation 2 : Stats par mois
stats_mensuel = (trips
    .join(broadcast(dates), "date_id")
    .groupBy("month", "year")
    .agg(
        count("*").alias("nb_trajets"),
        round(sum("total_amount"), 2).alias("revenu_total"),
        round(avg("trip_distance"), 2).alias("distance_moyenne")
    )
    .orderBy("year", "month")
)
stats_mensuel.show(12)
stats_mensuel.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_par_mois/")

In [ ]:
# Agregation 3 : Top 10 zones
top_zones = (trips
    .join(broadcast(locations),
          trips.pickup_location_id == locations.location_id)
    .groupBy("zone", "borough")
    .agg(count("*").alias("nb_trajets"))
    .orderBy(desc("nb_trajets"))
    .limit(10)
)
top_zones.show(truncate=False)
top_zones.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/top_zones/")

In [ ]:
# Agregation 4 : Impact meteo
stats_meteo = (trips
    .filter(col("weather_id") != -1)
    .join(broadcast(weather), "weather_id")
    .groupBy("weather_id", "temp_max", "precipitation", "snow")
    .agg(
        count("*").alias("nb_trajets"),
        round(avg("trip_distance"), 2).alias("distance_moyenne"),
        round(avg("total_amount"), 2).alias("tarif_moyen")
    )
)
stats_meteo.show()
stats_meteo.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_meteo/")

In [ ]:
# Agregation 5 : Matrice Origine-Destination (Top 20)
loc_pu = locations.withColumnRenamed("zone", "zone_pickup") \
    .withColumnRenamed("location_id", "pickup_location_id") \
    .select("pickup_location_id", "zone_pickup")

loc_do = locations.withColumnRenamed("zone", "zone_dropoff") \
    .withColumnRenamed("location_id", "dropoff_location_id") \
    .select("dropoff_location_id", "zone_dropoff")

od_matrix = (trips
    .join(broadcast(loc_pu), on="pickup_location_id")
    .join(broadcast(loc_do), on="dropoff_location_id")
    .groupBy("zone_pickup", "zone_dropoff")
    .agg(
        count("*").alias("nb_trajets"),
        round(avg("total_amount"), 2).alias("tarif_moyen")
    )
    .orderBy(desc("nb_trajets"))
    .limit(20)
)
od_matrix.show(truncate=False)
od_matrix.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/od_matrix/")

print("Silver -> Gold TERMINE")